# Section 5 — Observability ⭐ (priority section)

## Storyline

It's Tuesday morning. An Athora Netherlands team lead pings you: *"PolicyPal told a rep yesterday that policy NL-2031-887 had a balance of EUR 7,842.05 — it should be EUR 78,420.55. Find out what happened."*

Without traces you have: a screenshot from the rep, a vague *"around 10:15"*, and panic. With traces you have: the exact request, the exact tool call, the exact tool response, the exact model output, plus latency and token counts. **Observability is the difference between a 30-minute investigation and a 2-day one.**

## What you'll do

| Step | Concept | Why it matters for Athora Netherlands |
|------|---------|---------------------------|
| 1 | Enable OpenTelemetry locally (`configure_otel_providers`) | One-line setup. Traces appear in console / OTLP collector. |
| 2 | Send traces to **Foundry tracing UI** via Application Insights (`client.configure_azure_monitor`) | This is where Athora Netherlands' central platform team will look. |
| 3 | Add a custom span around a *scenario* | Group an entire rep call into one trace, end-to-end. |
| 4 | Inspect a trace in the Foundry portal (or App Insights) | What spans, attributes, and events you'll see. |

## Mental model

Agent Framework emits **OpenTelemetry** spans for: agent runs, tool calls, model calls, workflow executors. You don't write the instrumentation — you choose **where the spans go** (console, OTLP collector, App Insights / Foundry) and add business spans where Athora Netherlands needs auditability. Reference: [`microsoft/agent-framework — python/samples/observability`](https://github.com/microsoft/agent-framework/tree/main/python/samples/observability).

## Prereqs for this section

- An **Application Insights** resource linked to your Foundry project. (For the workshop, the instructor will share a connection.)
- `pip install azure-monitor-opentelemetry` (already in `requirements.txt`).
- Optional: a local OpenTelemetry Collector if you want to play offline.

## Local OpenTelemetry: see traces in the console

The fastest sanity check. Set `OTEL_TRACES_EXPORTER=console` (already in `.env.example`) and call `configure_otel_providers`. Every agent run, model call, and tool call now produces console spans.

In [ ]:
import os
from typing import Annotated
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.observability import configure_otel_providers, get_tracer
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id
from pydantic import Field

load_dotenv()
configure_otel_providers(enable_sensitive_data=True)
print("OTel providers configured (console exporter).")

**Note on `enable_sensitive_data=True`:** this includes prompts and tool arguments in spans. Great for debugging, **bad for production** if traces leave Athora Netherlands' perimeter. Set to `False` in any environment where you don't fully trust the trace destination.

In [ ]:
POLICY_BALANCES = {
    "NL-2031-887": "Jan de Vries — pension balance EUR 78,420.55",
    "NL-4408-552": "Sanne Bakker — term life insurance balance EUR 0.00; monthly premium EUR 42.50",
}

@tool(approval_mode="never_require")
def get_balance(policy_number: Annotated[str, Field(description="Athora Netherlands policy number")]) -> str:
    """Return the (mock) balance for a policy."""
    return POLICY_BALANCES.get(policy_number, f"No policy found for {policy_number}.")

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

policypal = Agent(
    client=client,
    name="PolicyPal",
    id="policypal-obs",
    instructions="You are PolicyPal. Use tools for any factual claim about a policy.",
    tools=[get_balance],
)

with get_tracer().start_as_current_span("Scenario: Rep call about NL-2031-887", kind=SpanKind.CLIENT) as span:
    print(f"Trace ID: {format_trace_id(span.get_span_context().trace_id)}")
    session = policypal.create_session()
    result = await policypal.run("What's the current balance on NL-2031-887?", session=session)
    print(f"PolicyPal: {result.text}")

### What you should see in the console

A burst of JSON span objects, including:

- **`Scenario: Rep call about NL-2031-887`** — your custom outer span.
- **`agent.run`** — the PolicyPal turn (with `agent.id`, `agent.name`).
- **`chat`** — the underlying model call (with model name, token counts, latency).
- **`tool.call`** — the `get_balance` invocation, with arguments and return value (because `enable_sensitive_data=True`).

Each child span links back to its parent via `trace_id`. That's how you reconstruct *what happened in this call*.

## Azure Monitor and Foundry tracing

`FoundryChatClient.configure_azure_monitor()` reads the App Insights connection string from the Foundry project and wires it up automatically. After this, traces show up in the **Foundry portal → Tracing** view, where the platform team will be looking.

In [ ]:
# Disable console exporter so configure_azure_monitor takes over.
os.environ.pop("ENABLE_CONSOLE_EXPORTERS", None)

client_with_monitor = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

await client_with_monitor.configure_azure_monitor(
    enable_sensitive_data=True,   # Athora Netherlands dev/test only
    enable_live_metrics=True,
)
print("Azure Monitor configured. Traces → app-insights-brk445 in Azure Portal.")

policypal_monitored = Agent(
    client=client_with_monitor,
    name="PolicyPal",
    id="policypal-obs",
    instructions="You are PolicyPal. Use tools for any factual claim about a policy.",
    tools=[get_balance],
)

with get_tracer().start_as_current_span("Athora Netherlands Rep Call", kind=SpanKind.CLIENT) as span:
    print(f"Trace ID: {format_trace_id(span.get_span_context().trace_id)}")
    session = policypal_monitored.create_session()
    span.set_attribute("athora.business_unit", "customer-service-nl")
    span.set_attribute("athora.regulators", "DNB,AFM")
    for q in ("Balance on NL-2031-887?", "And on NL-4408-552?", "Thanks, that's all."):
        print(f"\nrep: {q}")
        print("PolicyPal: ", end="")
        async for chunk in policypal_monitored.run(q, session=session, stream=True):
            if chunk.text:
                print(chunk.text, end="")
        print()

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.41.1",
                    "service.name": "agent_framework",
                    "service.version": "1.2.2"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "otel.sdk.processor.span.queue.size",
                            "description": "The number of spans in the queue of a given instance of an SDK span processor.",
                         

### What to look for in the Foundry tracing UI

Open Foundry → your project → **Tracing**. Filter by the trace ID printed above. You'll see:

- A **timeline view** with all spans nested under `Athora Netherlands Rep Call`.
- For each `chat` span: model name, prompt + completion tokens, **latency**.
- For each `tool.call` span: tool name, arguments, return value.
- The **agent ID** `policypal-obs` linking these spans to the agent registered in *Operate → Agents*.

If you don't see traces yet, give it ~30s — App Insights ingestion isn't instantaneous.

## Workflow-level observability

Workflows produce spans for each executor / agent participant in addition to the top-level workflow span. Re-run the **handoff workflow** from Section 4 with monitoring on, and inspect:

- One `workflow.run` span per call.
- Child `agent.run` spans, one per participant that actually ran (the coordinator and any specialist that received a handoff).
- Tool spans nested inside specialist `agent.run` spans.

This is exactly the data you'd want to answer *"why did this claim get routed to risk_agent instead of claims_agent?"* — the coordinator's `chat` span shows the model output that included the `transfer_to_risk_agent` tool call.

## Business span patterns

Use custom spans for business operations that matter outside engineering: a rep call, a policy lookup, a compliance review, or a handoff decision. Add low-cardinality attributes such as `athora.business_unit`, `policy.product_type`, or `regulatory_context`. Avoid customer names and raw policy text in production spans unless sensitive-data handling has been explicitly approved.

## Recap and what's next

- Tracing is **on by default** as soon as you call `configure_otel_providers` or `client.configure_azure_monitor`.
- Wrap each business *scenario* in a custom span so you can group traces meaningfully.
- For Athora Netherlands: use sensitive-data spans only in dev/test; turn them off (or scrub) before production.

**Coming up in Section 6 (Foundry Evaluations — priority topic):** traces tell you *what happened*. Evaluations tell you *whether it was good*. We'll run a small batch eval against PolicyPal with built-in evaluators (relevance, groundedness, …) plus a custom check.